In [47]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [48]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [49]:
len(words)

32033

In [50]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [51]:
# 划分数据集
block_size = 3
def build_dataset(words):
    X, Y = [], []

    for w in words:
        chs = list(w) + ['.']
        context = [0] * block_size
        for ch in chs:
            ix = stoi[ch]

            X.append(context)
            Y.append(ix)
            # print(''.join(itos[i] for i in context), '---->', itos[ix])
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)

n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


**now made respectful**

In [62]:
#MLP 
n_embd = 10
n_hidden = 200

g  = torch.Generator().manual_seed(1000) 
C  = torch.randn((vocab_size, n_embd),            generator= g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * 0.01
#b1 = torch.randn(n_hidden,                        generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0
bgain = torch.ones((1, n_hidden))
bias  = torch.zeros((1,n_hidden))
parameters = [C, W1, W2, b2, bgain, bias]

bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

for p in parameters:
    p.requires_grad = True

print(sum(p.nelement() for p in parameters))

stepi = []
lossi = []

12097


In [63]:
max_steps = 200000
batch_size = 32


for i in range(max_steps):
    #minibatch
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]

    # forward pass
    emb = C[Xb]
    #print(emb.shape)

    bh = emb.view(emb.shape[0],-1) @ W1 #+ b1

    bhmean = bh.mean(dim=0, keepdim=True)
    bhstd  = bh.std(dim=0, keepdim=True)
    #批量归一化
    bh = bgain * (bh - bhmean) / bhstd + bias
    
    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * bhmean
        bnstd_running  = 0.999 * bnstd_running  + 0.001 * bhstd

    h = torch.tanh(bh)

    logit = h @ W2 + b2
    # counts = logit.exp()
    # probs = counts / counts.sum(1,keepdim=True)
    # lossprob = -probs[torch.arange(32),Y].log().mean()
    loss = F.cross_entropy(logit, Yb)
    #print(f'loss = {loss.item()}')

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    lr = 0.1

    for p in parameters:
        p.data += -lr * p.grad

    if i % 10000 == 0:
        print(f'{i:7d}/{max_steps:7d}:{loss.item():.4f}')
    lossi.append(loss.log10().item())

print(f'loss = {loss.item()}')

      0/ 200000:3.6898
  10000/ 200000:2.2662
  20000/ 200000:2.2660
  30000/ 200000:2.4647
  40000/ 200000:2.2554
  50000/ 200000:2.2804
  60000/ 200000:1.8956
  70000/ 200000:2.3915
  80000/ 200000:2.0336
  90000/ 200000:2.0011
 100000/ 200000:2.8384
 110000/ 200000:2.4953
 120000/ 200000:2.2212
 130000/ 200000:1.9614
 140000/ 200000:1.9760
 150000/ 200000:2.2499
 160000/ 200000:2.2928
 170000/ 200000:2.1602
 180000/ 200000:1.8516
 190000/ 200000:1.9228
loss = 2.1951308250427246


In [57]:
@torch.no_grad()
def split_loss(split):
    x, y = {
        'train':(Xtr, Ytr),
        'val':(Xdev, Ydev),
        'test':(Xte, Yte)
    }[split]
    emb = C[x]
    embcat = emb.view(emb.shape[0], -1)
    bh = embcat @ W1 + b1
    bh = bgain * (bh - bnmean_running) / bnstd_running + bias
    h =  torch.tanh(bh)
    logit = h @ W2 + b2
    loss = F.cross_entropy(logit, y)
    print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.1363816261291504
val 2.1833865642547607


In [11]:
# 采样
g = torch.Generator().manual_seed(2001)

for _ in range(20):
    context = [0] * block_size
    out = []
    while True:
        emb = C[torch.tensor(context)]
        # print(emb.shape, C.shape, W1.shape)
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logit = h @ W2 + b2
        counts = logit.exp()
        probs = counts / counts.sum(dim=1, keepdim=True)
        ix = torch.multinomial(probs, num_samples=1, generator=g)

        ch = itos[ix.item()]
        context = context[1:] + [ix]
        out.append(ch)
        if ix == 0:
            break
    print(''.join(i for i in out))

cata.
zaia.
atialiyah.
atara.
oluwy.
jacki.
jakary.
rali.
ana.
jahavy.
joen.
junn.
leorcelaniyah.
salrosson.
kisuan.
fate.
jalexanna.
raylee.
jehsiella.
caria.
